# threshold-logic-unit

A tiny, faithful replication of the paper that **invented the artificial neuron** — built up **one idea at a time**. Run the cells top to bottom; each one adds a single piece.

> Warren S. McCulloch & Walter Pitts (1943). *A Logical Calculus of the Ideas Immanent in Nervous Activity.* The Bulletin of Mathematical Biophysics 5(4):115–133. [doi:10.1007/BF02478259](https://doi.org/10.1007/BF02478259)

**The big idea:** a neuron is *all-or-none* — it either fires or it doesn't. So “this neuron fired” is just a **true/false** statement, and a network of neurons becomes a circuit that computes **logic**.

## 1. The simplest possible neuron

Start with the least a neuron could do: **add up its inputs, and fire if the total is big enough.** That "big enough" line is the **threshold**.

No weights. No learning. Just counting and a threshold.

In [1]:
def neuron(inputs, threshold):
    total = sum(inputs)                     # add up the inputs that are ON (the 1s)
    return 1 if total >= threshold else 0   # fire only if the total reaches the threshold

Let's poke it. With a threshold of `2`, the total has to reach 2 before it fires:

In [2]:
print("both inputs on:", neuron([1, 1], threshold=2))
print("one input on:  ", neuron([1, 0], threshold=2))

both inputs on: 1
one input on:   0


## 2. Pick a threshold, get a logic gate

Here's the first surprise: **the threshold alone turns this neuron into different logic gates.**

- Threshold **2** → it only fires when *both* inputs are on → that's **AND**.
- Threshold **1** → *either* input is enough → that's **OR**.

In [3]:
def AND(a, b):
    return neuron([a, b], threshold=2)   # fires only if BOTH are on (total reaches 2)

def OR(a, b):
    return neuron([a, b], threshold=1)   # fires if EITHER is on (total reaches 1)

In [4]:
bits = [(0, 0), (0, 1), (1, 0), (1, 1)]   # every combination of two inputs

for name, gate in [("AND", AND), ("OR", OR)]:
    print(name)
    for a, b in bits:
        print(f"  {a} {b} -> {gate(a, b)}")

AND
  0 0 -> 0
  0 1 -> 0
  1 0 -> 0
  1 1 -> 1
OR
  0 0 -> 0
  0 1 -> 1
  1 0 -> 1
  1 1 -> 1


## 3. We hit a wall: NOT

Now try to build **NOT** — "fire when the input is *off*."

You can't. Adding inputs only ever pushes a neuron *toward* firing, never away from it. There's no threshold that means "fire when there's *less*."

Real neurons have a second kind of input that solves this: an **inhibitory** one. In this 1943 model it's absolute — **a single inhibitory signal vetoes firing entirely**, no matter the total. That's the one piece we add to the neuron.

In [5]:
def neuron(inputs, threshold, inhibited=False):
    if inhibited:                           # one inhibitory signal shuts it down completely
        return 0
    total = sum(inputs)
    return 1 if total >= threshold else 0

Now NOT is easy. A neuron with **threshold 0 and no excitatory inputs is "on by default"** (a total of 0 already meets a threshold of 0). The input's only job is to **inhibit** it — to switch it off.

In [6]:
def NOT(a):
    return neuron([], threshold=0, inhibited=bool(a))   # on by default; 'a' switches it off

print("NOT")
for a in (0, 1):
    print(f"  {a} -> {NOT(a)}")

NOT
  0 -> 1
  1 -> 0


## 4. Wire gates together → *any* logic (XOR)

One neuron has a famous limit: it **can't** do **XOR** ("one or the other, but not both"). No single threshold separates XOR's true cases from its false ones.

But that's fine — McCulloch & Pitts' real result is that **a *network* of these neurons can compute anything.** We already have the parts:

```
a XOR b  =  (a OR b)  AND  NOT(a AND b)
```

In [7]:
def XOR(a, b):
    return AND(OR(a, b), NOT(AND(a, b)))   # three gates we already built, wired together

print("XOR")
for a, b in bits:
    print(f"  {a} {b} -> {XOR(a, b)}")

XOR
  0 0 -> 0
  0 1 -> 1
  1 0 -> 1
  1 1 -> 0


## 5. Loop it → memory

Everything so far flows forward. Now **feed a neuron's own output back into itself.**

Give it a `set` input and a feedback input that is *its own previous output*. Once set, it keeps re-firing — it **reverberates** — so it *remembers* it was switched on, until a `reset` inhibits it. The authors call this firing “a memory — or an idea.”

First, one tick of time as a single neuron:

In [8]:
def memory_step(set_now, reset_now, was_on):
    # the neuron's inputs are the SET signal AND its own previous output (was_on).
    # that feedback -- its output looping back in -- is what lets it remember.
    return neuron([set_now, was_on], threshold=1, inhibited=bool(reset_now))

Now run it across several ticks of time. Watch the state turn on at `set`, **hold by itself**, then clear at `reset`:

In [9]:
# (set, reset) at each tick of time:  set at t=1, reset at t=4
ticks = [(0, 0), (1, 0), (0, 0), (0, 0), (0, 1), (0, 0)]

state = 0
print("t   set reset   state")
for t, (set_now, reset_now) in enumerate(ticks):
    state = memory_step(set_now, reset_now, state)
    print(f"{t}    {set_now}    {reset_now}       {state}")

t   set reset   state
0    0    0       0
1    1    0       1
2    0    0       1
3    0    0       1
4    0    1       0
5    0    0       0


## Self-check

A few assertions so the notebook proves itself — if any result disagreed with the paper, the cell would error.

In [10]:
assert [AND(a, b) for a, b in bits] == [0, 0, 0, 1]
assert [OR(a, b) for a, b in bits]  == [0, 1, 1, 1]
assert [NOT(a) for a in (0, 1)]     == [1, 0]
assert [XOR(a, b) for a, b in bits] == [0, 1, 1, 0]

states, s = [], 0
for set_now, reset_now in ticks:
    s = memory_step(set_now, reset_now, s)
    states.append(s)
assert states == [0, 1, 1, 1, 0, 0]

print("All checks pass -- one neuron, four ideas, true to 1943.")

All checks pass -- one neuron, four ideas, true to 1943.


## The whole neuron, in one place

We grew it across the notebook; here it is complete. That's the entire McCulloch–Pitts neuron:

```python
def neuron(inputs, threshold, inhibited=False):
    if inhibited:                           # absolute inhibition: one veto stops it
        return 0
    total = sum(inputs)                     # count the active excitatory inputs
    return 1 if total >= threshold else 0   # fire if the count meets the threshold
```

---

A network of these neurons is exactly a **finite-state machine** (logic + memory); add an external tape and it becomes **Turing complete** — the bridge from brains to computers, written in 1943. The next step in the story is [Rosenblatt's perceptron (1958)](https://doi.org/10.1037/h0042519): take this neuron, **add tunable weights and a learning rule**, and it can *learn from data* instead of being wired by hand.

*Educational reconstruction by [Average Joes Lab](https://averagejoeslab.com). All credit for the ideas to McCulloch & Pitts (1943).*